# 🎙️ VoiceNO — Silent Speech Recognition & Neural Disambiguation

**VoiceNO** enables silent, camera-only speech input on Google Colab GPU.

This master notebook runs the complete VoiceNO pipeline:
1. **Pretrained Auto-AVSR Conformer (GPU)**: Visual lip-reading backbone.
2. **In-Browser WebRTC Webcam Recorder**: Push-to-talk recording directly from your laptop camera into Colab.
3. **N-Best Beam Lattice Extraction**: Extracts visual phoneme candidates.
4. **Probabilistic Semantic Rescoring**: Disambiguates visually identical visemes ( -> *park at the mall*).
5. **Personalized LoRA Calibration**: Fine-tune a lightweight personal adapter in ~20 seconds on free T4 GPU.

---
### ⚡ Step 0: Ensure GPU is Enabled
Go to: **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> **Save**.

## 1. Verify GPU Hardware
Check that NVIDIA CUDA GPU is active.

In [ ]:
!nvidia-smi

## 2. Clone VoiceNO Repository & Install Dependencies

In [ ]:
import os
import sys

if not os.path.exists("VoiceNO"):
    !git clone https://github.com/keepsloading/VoiceNO.git

%cd VoiceNO

# Install required dependencies
!pip install -q av mediapipe opencv-python sentencepiece jiwer gradio gdown

# Add VoiceNO to Python module search path
sys.path.insert(0, os.getcwd())
print("VoiceNO environment ready!")

## 3. Download Auto-AVSR Base Model Checkpoint (~1.0 GB)
Downloads the 3,291-hour pretrained Visual Speech Recognition foundation weights.

In [ ]:
import os
from pathlib import Path

os.makedirs("checkpoints", exist_ok=True)
ckpt_path = Path("checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth")

if not ckpt_path.is_file():
    print("Downloading Auto-AVSR foundation weights...")
    !gdown 1r1kx7l9sWnDOCnaFHIGvOtzuhFyFA88_ -O checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth

print(f"Checkpoint ready at: {ckpt_path} (Size: {os.path.getsize(ckpt_path) / 1e6:.1f} MB)")

## 4. Initialize VoiceNO Pipeline on GPU
Loads the Conformer model and the probabilistic semantic rescorer.

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.io

# 1. OpenCV drop-in replacement for torchvision.io.read_video
def _read_video_cv2(filename, *args, **kwargs):
    cap = cv2.VideoCapture(filename)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise ValueError(f"Could not read frames from: {filename}")
    return torch.from_numpy(np.array(frames, dtype=np.uint8)), None, None

torchvision.io.read_video = _read_video_cv2

from voiceno.config import ModelConfig
from voiceno.models.auto_avsr import AutoAVSRModel
from voiceno.context.rescorer import ContextRescorer

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Loading VoiceNO on device: {device}...")

config = ModelConfig(device=device, beam_size=10)
model = AutoAVSRModel(config=config)
model.load_checkpoint("checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth")

rescorer = ContextRescorer()

print("
[48;5;22m[97m[1m MODEL & RESCORER READY! [0m
")

## 5. Live In-Browser Webcam Recorder (Push-to-Talk)
Because Colab runs in the cloud, this cell embeds a camera interface in your browser:
1. Run the cell below.
2. Click **🔴 Start Recording** and silently mouth your sentence (e.g. *"I want to park at the mall"* or *"I want to buy a kite"*).
3. Click **⏹️ Stop & Transcribe**.
4. VoiceNO processes the mouth movements and displays the N-best candidate lattice and the final disambiguated sentence!

In [ ]:
from google.colab import output
from IPython.display import display
import base64
import subprocess

def record_webcam(output_file="webcam_test.mp4"):
    js_code = """
    async function recordVideo() {
        const div = document.createElement("div");
        div.style.padding = "14px";
        div.style.border = "1px solid #444";
        div.style.borderRadius = "8px";
        div.style.background = "#181818";
        div.style.color = "#fff";
        div.style.width = "fit-content";
        
        const video = document.createElement("video");
        video.style.display = "block";
        video.style.borderRadius = "6px";
        video.width = 480;
        video.height = 360;
        video.autoplay = true;
        video.muted = true;
        video.playsInline = true;
        
        const btnRecord = document.createElement("button");
        btnRecord.innerText = "🔴 Start Recording";
        btnRecord.style.margin = "10px 8px 6px 0";
        btnRecord.style.padding = "8px 16px";
        btnRecord.style.fontWeight = "bold";
        btnRecord.style.cursor = "pointer";
        btnRecord.style.backgroundColor = "#d32f2f";
        btnRecord.style.color = "#fff";
        btnRecord.style.border = "none";
        btnRecord.style.borderRadius = "4px";

        const btnStop = document.createElement("button");
        btnStop.innerText = "⏹️ Stop & Transcribe";
        btnStop.disabled = true;
        btnStop.style.margin = "10px 8px 6px 0";
        btnStop.style.padding = "8px 16px";
        btnStop.style.fontWeight = "bold";
        btnStop.style.cursor = "pointer";
        btnStop.style.backgroundColor = "#555";
        btnStop.style.color = "#fff";
        btnStop.style.border = "none";
        btnStop.style.borderRadius = "4px";

        const status = document.createElement("span");
        status.innerText = "Initializing camera...";
        status.style.fontFamily = "monospace";

        div.appendChild(video);
        div.appendChild(btnRecord);
        div.appendChild(btnStop);
        div.appendChild(status);
        document.querySelector("#output-area").appendChild(div);

        const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 }, audio: false });
        video.srcObject = stream;
        status.innerText = " Ready. Click "Start Recording" and mouth words clearly.";

        const recorder = new MediaRecorder(stream, { mimeType: "video/webm" });
        const chunks = [];
        recorder.ondataavailable = e => { if (e.data.size > 0) chunks.push(e.data); };

        return new Promise((resolve) => {
            btnRecord.onclick = () => {
                chunks.length = 0;
                recorder.start();
                btnRecord.disabled = true;
                btnRecord.style.backgroundColor = "#555";
                btnStop.disabled = false;
                btnStop.style.backgroundColor = "#2e7d32";
                status.innerText = " 🔴 Recording... mouth your words now!";
            };

            btnStop.onclick = () => {
                recorder.stop();
                stream.getTracks().forEach(t => t.stop());
                btnStop.disabled = true;
                btnStop.style.backgroundColor = "#555";
                status.innerText = " ⏳ Resampling video to 25 FPS...";
            };

            recorder.onstop = () => {
                const blob = new Blob(chunks, { type: "video/webm" });
                const reader = new FileReader();
                reader.onloadend = () => resolve(reader.result);
                reader.readAsDataURL(blob);
            };
        });
    }
    recordVideo();
    """
    data_url = output.eval_js(js_code)
    header, encoded = data_url.split(",", 1)
    raw_bytes = base64.b64decode(encoded)
    
    with open("temp_webcam.webm", "wb") as f:
        f.write(raw_bytes)
        
    # Normalize framerate to 25.0 FPS MP4 (matching Conformer temporal expectation)
    subprocess.run([
        "ffmpeg", "-y", "-i", "temp_webcam.webm", "-r", "25",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", output_file
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    return output_file

video_clip = record_webcam("webcam_test.mp4")

# 1. Preprocess mouth ROI
print("Extracting normalized mouth ROI from video...")
tensor = model.preprocessor(video_clip)

# 2. Extract N-Best visual hypotheses
print("Running Conformer beam search decoding...")
details = model.transcribe_tensor(tensor, use_beam_search=True, return_details=True, nbest=5)
hypotheses = details.get("hypotheses", [])

# 3. Probabilistic LM Rescoring
rescored = rescorer.rescore_hypotheses(hypotheses)

print("
=======================================================")
print(f"[48;5;21m[97m TOP VISUAL HYPOTHESIS [0m: {hypotheses[0]["text"]}")
print(f"[48;5;28m[97m[1m FINAL RESCORED TEXT  [0m: {rescored[0]["text"]}")
print("=======================================================
")

print("--- Candidate Beam Lattice ---")
for i, h in enumerate(rescored):
    print(f"[{i+1}] {h["text"]} (Visual: {h["visual_score":.2f}, LM: {h["linguistic_score":.2f}, Total: {h["rescored_score":.2f}, Prob: {h.get("rescored_prob", 0)*100:.1f}%)")

## 6. (Optional) Run 1-Click LoRA Personal Calibration (~20s on GPU)
Mouth 8 guided calibration sentences to adapt the model to your personal facial anatomy and lip movements.

In [ ]:
from voiceno.calibration.trainer import PersonalizationTrainer

print("Starting LoRA Attention Personalization on GPU...")
trainer = PersonalizationTrainer(device=device, strategy="conformer_lora")

# Train 10 epochs (takes ~20 seconds on GPU)
metrics = trainer.train(user="default", epochs=10, lr=1e-3)
print(f"Personalization complete! Best loss: {metrics.get("best_loss", 0.0):.4f}")
print("Personalized profile saved in user_profiles/default/lora.pt")

## 7. Launch Interactive VoiceNO Web UI
Launches a web interface with a shareable public link.

In [ ]:
!python -m app.main ui --share